# 01 — Data Exploration

Purpose: Understand raw dataset characteristics (distribution, durations, sample rates), run quality checks (clipping/silence), and produce baseline visualizations for angle_grinder, background, and tools classes.

In [1]:
# Imports and setup
import os
from pathlib import Path
import yaml
import sys
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display as ld
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

plt.style.use("seaborn-v0_8")
sns.set_context("notebook")

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = PROJECT_ROOT / "results" / "figures"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Figures:", FIG_DIR)
print("Metrics:", METRICS_DIR)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4
Figures: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures
Metrics: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/metrics


In [2]:
cfg_path = PROJECT_ROOT / "config.yaml"
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

# Accept either:
# -  { raw_dir, processed_dir, ... }
# - config: { raw_dir, processed_dir, ... }  (your current variant)
data_cfg = (
    cfg.get("data")
    or cfg.get("config", {}).get("data")
    or cfg.get("config")
    or {}
)

raw_dir = PROJECT_ROOT / data_cfg.get("raw_dir", "data/raw")
processed_dir = PROJECT_ROOT / data_cfg.get("processed_dir", "data/processed")
augmented_dir = PROJECT_ROOT / data_cfg.get("augmented_dir", "data/augmented")

print("Raw dir:", raw_dir)
assert raw_dir.exists(), f"Raw data directory not found: {raw_dir}"


Raw dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/raw


In [3]:
# Enumerate dataset files
AUDIO_EXTS = (".wav", ".flac", ".mp3", ".m4a", ".ogg")
CLASSES = ["angle_grinder", "background", "tools"]

records = []
for cls in CLASSES:
    cls_dir = raw_dir / cls
    if not cls_dir.exists():
        print(f"Warning: missing class directory: {cls_dir}")
        continue
    for p in cls_dir.rglob("*"):
        if p.suffix.lower() in AUDIO_EXTS and p.is_file():
            records.append({"path": p, "label": cls})

df = pd.DataFrame(records)
print("Found files:", len(df))
df.head()


Found files: 1405


,path,label
0,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder
1,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder
2,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder
3,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder
4,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder


In [4]:
# Extract metadata and basic measures
def get_audio_info(path):
    try:
        info = sf.info(path)
        sr = info.samplerate
        channels = info.channels
        subtype = info.subtype
    except Exception:
        sr, channels, subtype = None, None, None
    try:
        dur = librosa.get_duration(path=str(path))
    except Exception:
        dur = None
    return sr, channels, subtype, dur

meta = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    sr, ch, sub, dur = get_audio_info(row["path"])
    meta.append({"samplerate": sr, "channels": ch, "subtype": sub, "duration_s": dur})

meta_df = pd.DataFrame(meta)
df_meta = pd.concat([df.reset_index(drop=True), meta_df], axis=1)
df_meta.head()


  0%|          | 0/1405 [00:00<?, ?it/s]

,path,label,samplerate,channels,subtype,duration_s
0,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,48000,2,PCM_16,37.740000
1,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,24000,2,MPEG_LAYER_III,37.800000
2,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,44100,2,PCM_24,166.504490
3,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,48000,1,PCM_24,41.335813
4,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,44100,2,MPEG_LAYER_III,41.501905


In [5]:
# Quality checks (clipping and silence)
def analyze_quality(path, clip_thresh=0.99, top_db=20):
    try:
        y, sr = librosa.load(path, sr=None, mono=True)
        if y.size == 0:
            return True, 100.0
        peak = np.max(np.abs(y))
        is_clipped = bool(peak >= clip_thresh)
        intervals = librosa.effects.split(y, top_db=top_db)
        non_silent = sum((e - s) for s, e in intervals)
        silence_ratio = float(max(0, len(y) - non_silent)) / len(y) * 100.0
        return is_clipped, silence_ratio
    except Exception:
        return None, None

qc = []
for _, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    ic, srp = analyze_quality(row["path"])
    qc.append({"clipped": ic, "silence_pct": srp})

qc_df = pd.DataFrame(qc)
df_full = pd.concat([df_meta.reset_index(drop=True), qc_df], axis=1)
df_full.head()


  0%|          | 0/1405 [00:00<?, ?it/s]

,path,label,samplerate,channels,subtype,duration_s,clipped,silence_pct
0,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,48000,2,PCM_16,37.740000,False,2.571984
1,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,24000,2,MPEG_LAYER_III,37.800000,False,2.539683
2,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,44100,2,PCM_24,166.504490,False,47.808807
3,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,48000,1,PCM_24,41.335813,False,2.328439
4,/Users/harryirving/Development/projects/ai-ml/...,angle_grinder,44100,2,MPEG_LAYER_III,41.501905,False,5.482042


In [6]:
# Aggregate statistics and save CSV
summary = {
    "n_files": len(df_full),
    "by_class": df_full.groupby("label")["path"].count().to_dict(),
    "duration_total_s": float(df_full["duration_s"].dropna().sum()),
    "duration_mean_s": float(df_full["duration_s"].dropna().mean()) if df_full["duration_s"].notna().any() else None,
    "samplerates": df_full["samplerate"].value_counts(dropna=False).to_dict(),
    "clipped_pct": float(np.mean([x for x in df_full["clipped"] if x is not None]) * 100.0) if df_full["clipped"].notna().any() else None,
    "median_silence_pct": float(df_full["silence_pct"].dropna().median()) if df_full["silence_pct"].notna().any() else None,
}
summary_df = pd.DataFrame([summary])
out_csv = METRICS_DIR / "dataset_summary.csv"
df_full.to_csv(METRICS_DIR / "dataset_files_detailed.csv", index=False)
summary_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
summary_df.T


Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/metrics/dataset_summary.csv


,0
n_files,1405
by_class,"{'angle_grinder': 36, 'background': 1348, 'too..."
duration_total_s,7300.344656
duration_mean_s,5.195975
samplerates,"{16000: 1346, 44100: 25, 48000: 18, 24000: 16}"
clipped_pct,1.779359
median_silence_pct,0.0


In [7]:
# Visualizations (class balance, durations, sample rates)
def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    print("Saved figure:", path)

# Class distribution
plt.figure(figsize=(5,4))
sns.countplot(data=df_full, x="label", order=CLASSES)
plt.title("Class Distribution")
savefig(FIG_DIR / "class_distribution.png")
plt.close()

# Duration distribution
plt.figure(figsize=(6,4))
sns.histplot(df_full["duration_s"].dropna(), bins=40, kde=True)
plt.xlabel("Duration (s)")
plt.title("Duration Distribution")
savefig(FIG_DIR / "duration_distribution.png")
plt.close()

# Sample rate distribution
plt.figure(figsize=(6,4))
sr_counts = df_full["samplerate"].value_counts(dropna=False)
sr_counts.plot(kind="bar")
plt.ylabel("Count")
plt.title("Sample Rate Distribution")
savefig(FIG_DIR / "samplerate_distribution.png")
plt.close()


Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/class_distribution.png
Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/duration_distribution.png
Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/samplerate_distribution.png


In [8]:
def plot_example(path, title_prefix=""):
    y, sr = librosa.load(path, sr=None, mono=True)
    fig, ax = plt.subplots(2, 1, figsize=(8,5))
    ld.waveshow(y, sr=sr, ax=ax[0], color='blue')  # Add color parameter
    ax[0].set_title(f"{title_prefix}Waveform — sr={sr}")

    S = librosa.stft(y, n_fft=2048, hop_length=512)
    S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
    img = ld.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="log", ax=ax[1], cmap="magma")
    ax[1].set_title(f"{title_prefix}Spectrogram (log-freq)")
    fig.colorbar(img, ax=ax[1], format="%+2.0f dB")
    return fig



In [9]:
print("Files by class:\n", df_full.groupby("label")["path"].count(), "\n")
print("Sample rate distribution:\n", df_full["samplerate"].value_counts(dropna=False), "\n")
print("Mean duration (s):", df_full["duration_s"].dropna().mean())
print("Clipped ratio:", np.mean([x for x in df_full['clipped'] if x is not None]))
print("Median silence %:", df_full["silence_pct"].dropna().median())


Files by class:
 label
angle_grinder      36
background       1348
tools              21
Name: path, dtype: int64 

Sample rate distribution:
 samplerate
16000    1346
44100      25
48000      18
24000      16
Name: count, dtype: int64 

Mean duration (s): 5.195974844356485
Clipped ratio: 0.017793594306049824
Median silence %: 0.0
